# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Loading Necessary Tables

In [0]:
customer_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_customer`""")
country_master_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_country_master`""")
fx_rate_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_fx_rate_df`""")
opportunity_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_opportunity`""")

# Business Logics

In [0]:
opportunity_df=opportunity_df.alias("o").\
    join(customer_df.alias("c"), 
        on=col("o.customer_id") == col("c.customer_id"), 
        how="left")\
    .select("o.*", "c.country").alias("co")\
    .join(country_master_df.alias("cm"),
        on=col("co.country") == col("cm.country_code"), 
        how="left")\
    .select("co.*", "cm.currency_code").alias("coc")\
    .join(fx_rate_df.alias("fx"),
        on=col("coc.currency_code") == col("fx.currency_code"),
        how="left")\
    .select("coc.*", "fx.fx_rate_to_gbp")

In [0]:
opportunity_df=opportunity_df.withColumn("revenue_in_gpb",col("revenue_amount")*col("fx_rate_to_gbp"))

In [0]:
opportunity_df = opportunity_df.withColumn(
    "contract_months",
    when(col("contract_term") == "Monthly", 1)
    .when(col("contract_term") == "Yearly", 12)
)
opportunity_df = opportunity_df.withColumn(
    "mrr_in_gpb",
    col("revenue_in_gpb") / col("contract_months")
)

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

### Applying SCD Type 1 for Subscription Table

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{SILVER_SCHEMA_PATH}.silver_subscription"):
    dt = DeltaTable.forName(spark, f"{SILVER_SCHEMA_PATH}.silver_subscription")
    dt.alias("target").merge(
        opportunity_df.alias("source"),
        "target.opportunity_id = source.opportunity_id" 
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    opportunity_df.write.format("delta").saveAsTable(f"{SILVER_SCHEMA_PATH}.silver_subscription")

# Optimizing Subscription Table

In [0]:
spark.sql(f"""
OPTIMIZE {SILVER_SCHEMA_PATH}.`silver_subscription`
ZORDER BY (customer_id, product_id)""")